# 01 · GitHub Copilot SDK 快速上手

目标：在 10 分钟内跑通第一个 SDK 调用，理解 `CopilotClient` / `CopilotSession` 的生命周期。

> 状态：**Public Preview**，API 可能会 breaking change。
> 文档：<https://github.com/github/copilot-sdk/tree/main/python>

## 0.0 GHCP SDK 核心概念速查

> 在动手前先对齐这 13 个术语，下面所有 cell 都会反复出现。所有名称均**已核对自 `copilot` 包源码**（无杜撰）。

| # | 概念 | 出处 / API | 一句话定义 |
|---|---|---|---|
| 1 | **`CopilotClient`** | `copilot.CopilotClient` | SDK 的入口对象；启动后会 fork 一个 `copilot` CLI 子进程并与之 JSON-RPC 通讯 |
| 2 | **`CopilotSession`** | `copilot.session.CopilotSession` | 一次"对话"的句柄；一个 client 下可创建多个，互相隔离（独立 `session_id` / history / `working_directory`） |
| 3 | **`session_id`** | `session.session_id: str` | 服务端给每个 session 分配的 UUID；磁盘持久化目录就以它命名 |
| 4 | **JSON-RPC** | stdin / stdout 文本协议 | `method` + `params` + `id` 的请求/响应；通知（无 `id`）用于服务端主动推事件 |
| 5 | **事件 / `SessionEvent`** | `copilot.generated.session_events.*Data` | 子进程推给你的所有变化；用 `session.on(handler)` 订阅，按 `event.data` 类型 `match` |
| 6 | **`SessionIdleData`** | 同上 | **本轮真正结束**的唯一可靠信号——不要数 `AssistantMessage` 条数 |
| 7 | **`AssistantMessageData` / `.delta`** | 同上 | 模型回复（完整段 / 流式增量）；一轮可能有**多条** |
| 8 | **Permission Gate** | `on_permission_request=` + `PermissionHandler` | v2 协议必传；不传则所有工具调用永远卡住等裁决 |
| 9 | **Tool / `@define_tool`** | `copilot.tool.define_tool` | 把 Python 函数注册成模型可调用的工具；handler 在你这边执行 |
| 10 | **Provider / BYOK** | `provider={'type': 'azure'/...}` | 绕过 GitHub 鉴权，用你自己的 Azure OpenAI / 其它模型端点；**每次 resume 都要重传** |
| 11 | **`system_message=`** | `copilot.session` 的 `SystemMessageConfig` | 三种模式：`append` / `replace` / `customize`（按 10 个 prompt section 精修） |
| 12 | **`SessionFsProvider`** | `copilot.session_fs_provider` + `create_session_fs_adapter` | 给 agent 一个**虚拟文件系统**接口（10 个 async 方法），绕开真实磁盘 |
| 13 | **`sessions.fork`** | `client.rpc.sessions.fork(...)`（实验性） | 在某个事件之前把 session 切两条分支，做 A/B 推演 |

> 物理布局：`~/.copilot/session-state/<session_id>/` 下有 `events.jsonl`（真相之源）+ `workspace.yaml`（cwd/git/branch 元数据）+ `checkpoints/` + `files/`。详见 [docs/session-persistence.md](docs/session-persistence.md)。


## 0. SDK 全景架构：**GHCP SDK ↔ Agent Harness Engineering 映射**

> 这份 SDK 的全部表面积，本质是 **agent harness engineering** 这门工程的标准工位。
> 与其在 markdown 里堆图，不如直接看那张为映射而专门画的独立架构图：

📐 **[docs/harness-mapping.html](docs/harness-mapping.html)** — 10 条 Harness 关注点 ↔ SDK API 一一对应映射图

里面回答的唯一问题：**SDK 暴露的每个参数 / 方法，到底解决 harness 工程里哪个具体问题？**

- 左列：harness 工程理念（"为什么需要"）
- 右列：SDK 的具体 API 钩子（"在哪里改"）
- 底部：把所有参数收束进一张 6 节点的 **agentic loop** 图（Context Build → LLM → Permission Gate → Tool Exec → Observation → Compaction → 回到 Context）

> 💡 在 VS Code 里：在文件资源管理器右键 `docs/harness-mapping.html` → **Open Preview**（或浏览器打开）查看。
> 也可以执行下一个 cell，把它嵌进 notebook 直接看。

**一句话论点**：GHCP SDK 不是 LLM client，它是把 "**agentic loop + 边界控制 + 持久化**" 这套 harness 工程做成产品的薄壳。你写业务代码时调的每个参数，本质都是在 harness 的标准工位上拧一颗螺丝。


In [ ]:
# 💡 在 notebook 里直接嵌入查看 harness-mapping.html
from IPython.display import IFrame
IFrame(src='docs/harness-mapping.html', width='100%', height=900)


## 0.1 一次 `send` 的完整时序

「用户发 1 句话 → 模型调 1 个工具 → 给最终答复」这种典型一轮的时序图。理解这张图，事件订阅、权限、`send_and_wait` 全都顺：

```mermaid
sequenceDiagram
    autonumber
    participant App as 你的代码
    participant Sess as CopilotSession
    participant CLI as copilot CLI 子进程
    participant LLM as Provider<br/>(Azure OpenAI 等)
    participant Tool as 工具 handler<br/>(@define_tool)

    App->>Sess: await session.send("prompt")
    Sess->>CLI: JSON-RPC: session.send
    CLI-->>Sess: event: user.message
    CLI->>LLM: chat.completions
    LLM-->>CLI: assistant message (含 tool_call)
    CLI-->>Sess: event: assistant.message.delta (streaming=True 时)
    CLI-->>Sess: event: assistant.message
    CLI->>App: on_permission_request(req)
    App-->>CLI: PermissionRequestResult(approve-once)
    CLI-->>Sess: event: tool.executionStart
    CLI->>Tool: 调用 handler
    Tool-->>CLI: 返回结果
    CLI-->>Sess: event: tool.executionComplete
    CLI->>LLM: 把 tool 结果回灌给模型
    LLM-->>CLI: 最终自然语言回复
    CLI-->>Sess: event: assistant.message (final)
    CLI-->>Sess: event: session.idle
    Sess-->>App: done.set() / send_and_wait 返回
```

**3 个一定要记住的坑**（详见 [README.md](README.md) §5）：

1. `await session.send(...)` 返回的是 **message ID**（UUID 字符串），**不是**回复文本。要文本请用 `await session.send_and_wait(...)` 或订阅 `AssistantMessageData` 事件。
2. 协议 v2 强制要求 `on_permission_request`，没有它工具调用会**永远卡在权限请求**。最简放行：`PermissionHandler.approve_all`。
3. `SessionIdleData` 是**唯一可靠**的「本轮结束」信号，不要靠 `assistant.message` 数次数（一轮可能有多条）。


## 1. 安装

- Python ≥ 3.11
- `github-copilot-sdk` 会自动 bundle `copilot` CLI 二进制，**不需要**单独安装 CLI
- 顶层模块名为 `copilot`

认证方式（任一）：
1. `COPILOT_GITHUB_TOKEN` / `GH_TOKEN` / `GITHUB_TOKEN` 环境变量
2. 已用 `copilot` CLI `login` 过的本地账号
3. **BYOK**：传 `provider={...}`，绕过 GitHub 鉴权（见 notebook 05）

In [ ]:
%pip install -q github-copilot-sdk python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# 从指定绝对路径加载 .env
load_dotenv()

# 提取并配置 Azure OpenAI GPT 5.4 服务提供商凭据以进行 BYOK 认证
azure_provider = {
    'type': 'azure',
    'base_url': os.environ['AZURE_OPENAI_ENDPOINT_GPT_5_4'],
    'api_key': os.environ['AZURE_OPENAI_API_KEY_GPT_5_4'],
    'azure': {'api_version': os.getenv('AZURE_OPENAI_API_VERSION_GPT_5_4', '2025-03-01-preview')},
}

## 2. 最小可运行示例（async context manager 写法 — 推荐）

套路：
1. `async with CopilotClient() as client` —— 自动启动 / 关闭 CLI 子进程
2. `async with await client.create_session(...) as session` —— 自动 destroy
3. `session.on(handler)` 订阅事件 → `session.send(prompt)` → 等 `SessionIdleData`

`on_permission_request` 在 v2 协议下**必须**显式提供（即使用 `PermissionHandler.approve_all`），否则每次工具调用都会卡住等待外部裁决。

### 2.1 `CopilotClient` / `CopilotSession` 生命周期

```mermaid
stateDiagram-v2
    [*] --> ClientStopped: import copilot

    state "CopilotClient" as Client {
        ClientStopped --> ClientStarted: await client.start()<br/>(或 async with)
        ClientStarted --> ClientStopped: await client.stop()
        ClientStarted --> ClientStarted: 创建/销毁 N 个 session
    }

    state "CopilotSession (单个)" as Session {
        [*] --> Created: await client.create_session(...)
        Created --> Idle: 协议握手 + system_message 注入
        Idle --> Busy: session.send(prompt)
        Busy --> Busy: event: assistant.message.delta /<br/>tool.executionStart / ...
        Busy --> Idle: event: session.idle
        Busy --> Aborted: await session.abort()
        Aborted --> Idle
        Idle --> Destroyed: session.disconnect() /<br/>__aexit__
        Destroyed --> [*]
    }

    ClientStarted --> Created
```

记住：**`Idle` 才是可以再次发 `send` 的状态**。在 `Busy` 期间调 `send` 不会报错，但消息会排队，事件可能交错—正确做法是 `await done.wait()` 或 `send_and_wait`，等收到 `SessionIdleData` 再发下一条。


In [ ]:
import asyncio
from copilot import CopilotClient
from copilot.session import PermissionHandler
from copilot.generated.session_events import AssistantMessageData, SessionIdleData

async def hello():
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
        ) as session:
            done = asyncio.Event()
            collected: list[str] = []

            def on_event(event):
                match event.data:
                    case AssistantMessageData() as d:
                        collected.append(d.content)
                    case SessionIdleData():
                        done.set()

            session.on(on_event)
            await session.send('用一句话解释 GitHub Copilot SDK 是什么。')
            await done.wait()
            return '\n'.join(collected)

print(await hello())  # Jupyter 顶层 await 可用

## 3. 手动生命周期（需要细粒度控制时）

**什么场景用？** 当 session 的存活范围**跨越** Python 的 `with` 作用域时，比如：

- 长驻服务（FastAPI / Flask 后端）—— session 跟着 HTTP 请求生命周期，不能塞进 `async with`
- 需要在创建后**先注册 handler 再 send**，并保留 session 对象供后续轮次复用
- 需要在 `disconnect` 之前手动 `abort()` 一个正在跑的长任务

记住三件事：

1. `await client.start()` / `await client.stop()` 必须成对出现，否则 CLI 子进程会泄漏
2. `await session.disconnect()` 只是关闭这一个 session 的 RPC 通道，**不会**停掉 client
3. 任何异常路径都要走 `finally`——`async with` 的本质就是帮你写这段


In [ ]:
from copilot import CopilotClient
from copilot.session import PermissionHandler
from copilot.generated.session_events import AssistantMessageData, SessionIdleData

client = CopilotClient()
await client.start()
session = await client.create_session(
    on_permission_request=PermissionHandler.approve_all,
    model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
    provider=azure_provider,
)
try:
    # 1) 先注册事件 handler
    collected: list[str] = []
    idle = asyncio.Event()

    def on_event(event):
        match event.data:
            case AssistantMessageData() as d:
                collected.append(d.content)
            case SessionIdleData():
                idle.set()

    unsubscribe = session.on(on_event)

    # 2) 多次 send，每轮 wait idle —— 复用同一个 session
    for prompt in ['你好', '把上一句翻成英文']:
        idle.clear()
        await session.send(prompt)
        await idle.wait()

    print('\n---\n'.join(collected))

    # 3) 不再需要这个 handler 时可取消订阅
    unsubscribe()
finally:
    await session.disconnect()
    await client.stop()


## 4. 一次性 helper：`send_and_wait`

如果你**只关心最终回复**、不在乎中间的 streaming delta / tool 事件，用 `send_and_wait` 一行搞定，省掉自己写 `Event + handler + done.wait()` 的样板代码。

**真实签名**（核对自 SDK 源码 `copilot/session.py`）：

```python
async def send_and_wait(
    prompt: str,                       # 注意是 str，不是 dict
    *,
    attachments: list[Attachment] | None = None,
    mode: Literal['enqueue', 'immediate'] | None = None,
    request_headers: dict[str, str] | None = None,
    timeout: float = 60.0,
) -> SessionEvent | None                # 最后一条 AssistantMessage 事件，没有则 None
```

- **返回值**：`SessionEvent | None`，其 `.data` 是 `AssistantMessageData`；整轮若没产生 assistant message（极少见）则 `None`
- **`timeout=` 的语义**：只控制"等多久不收到 `session.idle` 就抛 `TimeoutError`"，**不会** abort 正在跑的 agent —— 超时后 agent 仍在 CLI 子进程里继续跑，要真正打断必须 `await session.abort()`

### 4.1 最小例子


In [ ]:
from copilot import CopilotClient
from copilot.session import PermissionHandler
from copilot.generated.session_events import AssistantMessageData

async def one_shot(prompt: str) -> str:
    async with CopilotClient() as client:
        async with await client.create_session(
            on_permission_request=PermissionHandler.approve_all,
            model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
            provider=azure_provider,
        ) as session:
            event = await session.send_and_wait(prompt, timeout=120.0)
            if event is None:
                return '(no reply)'
            match event.data:
                case AssistantMessageData() as d:
                    return d.content
            return repr(event.data)

print(await one_shot('用三句话解释什么是 JSON-RPC。'))

### 4.2 超时 + 中断

```python
try:
    event = await session.send_and_wait('帮我跑个超复杂的任务', timeout=10.0)
except TimeoutError:
    # 仅仅是「等不及了」，agent 还在 CLI 那边跑
    await session.abort()   # 这才真的把它停掉
```

`abort()` 之后 session 还活着，可以继续 `send` 下一轮——它打断的是当前 turn，不是 session。

## 4.3 多 session 并发演示

证明 takeaway #7：**同一个 `CopilotClient` 下并发跑 N 个 session，各自一份 `session_id` / `history` / `working_directory`，互不串扰**。

下面这段同时跑 3 个 session：
- 同时 `asyncio.gather` 三个 `send_and_wait`
- 每个 session 传不同的 `working_directory=`
- 打印每个 session 的 `session_id` + 回复，可以看到 ID 不同、回复彼此独立


In [ ]:
import asyncio, tempfile, pathlib
from copilot import CopilotClient
from copilot.session import PermissionHandler
from copilot.generated.session_events import AssistantMessageData

async def one_session(client, prompt: str, cwd: str):
    async with await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=os.environ.get('AZURE_OPENAI_MODEl_GPT_5_4', 'gpt-5.4'),
        provider=azure_provider,
        working_directory=cwd,
    ) as session:
        sid = session.session_id
        event = await session.send_and_wait(prompt, timeout=120.0)
        reply = ''
        if event is not None:
            match event.data:
                case AssistantMessageData() as d:
                    reply = d.content
        return sid, cwd, reply

async def concurrent_demo():
    # 给每个 session 一个独立的临时工作目录
    cwds = [tempfile.mkdtemp(prefix=f'sess{i}_') for i in range(3)]
    prompts = [
        '用一句话说说你是谁。',
        '1 + 1 = ?  只回答数字。',
        '把 "hello world" 翻成中文，只给翻译。',
    ]
    async with CopilotClient() as client:
        results = await asyncio.gather(*[
            one_session(client, p, c) for p, c in zip(prompts, cwds)
        ])
    for sid, cwd, reply in results:
        print(f'\nsession_id = {sid}')
        print(f'cwd        = {cwd}')
        print(f'reply      = {reply}')

await concurrent_demo()


## 5. 关键 takeaways

| # | 要点 | 反面教材 |
|---|---|---|
| 1 | SDK = Python API + bundled `copilot` CLI 子进程，通过 **JSON-RPC over stdin/stdout** 通讯 | 把它当 OpenAI SDK 用、想直接传 `messages=[...]` |
| 2 | 永远用 `async with CopilotClient()` / `async with await client.create_session(...)` | 忘了 `disconnect` / `stop`，子进程泄漏直到 OOM |
| 3 | `await session.send(...)` 返回的是 **message ID（str）**，不是回复文本 | `print(await session.send('...'))` 然后困惑为啥是 UUID |
| 4 | 一轮可能有**多条** `AssistantMessage`（多个 tool 调用之间会插自然语言）；只有 `SessionIdleData` 标志该轮真正结束 | 用 `assistant.message` 次数判断结束，复杂任务直接死锁 |
| 5 | 协议 v2 要求显式 `on_permission_request=`；没传会卡死 | 生产请自己实现策略；最省事的放行用 `PermissionHandler.approve_all` |
| 6 | `timeout=` 只是"等多久就放弃等"，**不会**真的中断 agent；中断必须 `await session.abort()` | 超时后直接发下一条，新旧 turn 事件混在一起 |
| 7 | 多个 session 可并发，互不影响（各自一份 history & working_directory） | 想用全局变量在 session 间共享状态 |

### 下一步

| Notebook | 主题 |
|---|---|
| **02** | 事件类型全景 + 流式输出 (`assistant.message.delta`) |
| **03** | 自定义工具 (`@define_tool`) |
| **04** | 权限策略 + hooks |
| **05** | BYOK / 多 provider |
| **09** | `system_message` 三种模式 / `SessionFsProvider` / `sessions.fork` |